# Foundation Strategy: Return Predictability + Volatility Timing
### Applying Lectures 5.6 and 5.7 to Kalshi Prediction Markets

This notebook replicates the full walk-forward backtest of the Foundation Strategy,
including all empirical results, sub-sample analyses, robustness checks, figures, and tables.

**Strategy summary:**
Two techniques from finance lecture notes are applied to prediction markets:
1. **Return Predictability (Lecture 5.6):** An OLS regression slope over a lookback window estimates the expected daily price change (μ̂) — the prediction-market analog of the dividend yield forecasting regression.
2. **Volatility Timing (Lecture 5.7):** Positions are scaled by inverse realized variance (1/RV) — invest more when volatility is low, less when it is high.

Combined mean-variance optimal weight:
$$w = \frac{\hat{\mu}}{\gamma \cdot RV}$$

**Data:** Live market data from the [Kalshi](https://kalshi.com) prediction market API — no file upload needed.

**Runtime:** Fetching 180 days of history for 200 markets takes ~5–8 minutes.

In [ ]:
# Install dependencies (run once)
!pip install -q requests pandas scipy numpy tabulate colorama matplotlib

In [ ]:
import time
import warnings
from datetime import datetime, timezone
from typing import Optional

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from tabulate import tabulate

warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 9

print('Imports OK')

In [ ]:
# ── Strategy parameters ───────────────────────────────────────────────────────
# These match the final configuration documented in model.py.

GAMMA       = 2.0    # Risk aversion γ. At γ=2 the weight cap binds on every trade,
                     # giving equal-sized positions (see design note 6).
LOOKBACK    = 30     # Days of price history used for OLS + RV estimation.
MIN_R2      = 0.10   # Minimum OLS R² to trust the signal (design note 3).
MAX_WEIGHT  = 0.25   # Cap on raw weight before normalization (design note 6).
MIN_HISTORY = 10     # Minimum data points required to run estimate().
MAX_SPREAD  = 0.10   # Skip markets with bid-ask spread > 10% of mid-price.
MIN_PRICE   = 0.15   # Skip trades where entry price < 15% (design note 14).
                     # Markets priced below 0.15 have a negative realized Sharpe
                     # (-0.34) — the mean-reversion signal is structurally weakest
                     # for markets already near their floor.

HOLD_PERIOD = 7      # Hold period in days for the backtest.
FETCH_DAYS  = 180    # Days of price history to fetch per market.
LIMIT       = 200    # Max markets to process, ranked by volume.

BASE_URL = 'https://api.elections.kalshi.com/trade-api/v2'
HEADERS  = {'Accept': 'application/json'}

# Sports markets follow momentum, not mean-reversion (design note 10).
# Every sports market in backtesting was a consistent loser.
SPORTS_KEYWORDS = [
    'nba', 'nfl', 'nhl', 'mlb', 'nascar', 'pga', 'premier',
    'champions', 'bundesliga', 'laliga', 'seriea', 'ligue1',
    'mvp', 'championship', 'playoff', 'superbowl', 'worldseries'
]

print('Configuration set.')

## 1. Data Access — Kalshi Prediction Market API

Data is fetched live from the Kalshi public API. No authentication required for read access.
- `fetch_events()` pages through all open events across all categories.
- `fetch_market_history()` retrieves daily candlestick close prices for a given market.
- Sports markets are excluded by keyword match on the ticker/title before history is fetched.

In [ ]:
def kalshi_get(path: str, params: dict = {}) -> dict:
    url = f"{BASE_URL}/{path.lstrip('/')}"
    resp = requests.get(url, headers=HEADERS, params=params, timeout=15)
    resp.raise_for_status()
    return resp.json()


def to_float(val) -> Optional[float]:
    """Parse a value to float and validate it is a probability in [0, 1]."""
    try:
        f = float(val)
        return f if 0.0 <= f <= 1.0 else None
    except (TypeError, ValueError):
        return None


def fetch_events() -> list:
    """Page through all open events across all Kalshi categories."""
    events, cursor, page = [], None, 0
    print('Fetching all open events…', end='', flush=True)
    while True:
        params = {'limit': 100, 'status': 'open', 'with_nested_markets': 'true'}
        if cursor:
            params['cursor'] = cursor
        data   = kalshi_get('events', params)
        batch  = data.get('events', [])
        events.extend(batch)
        page  += 1
        cursor = data.get('cursor')
        print(f'\r  Page {page}: {len(events)} events…', end='', flush=True)
        if not cursor or not batch:
            break
        time.sleep(0.2)
    print(f'\r  Done — {len(events)} events fetched.          ')
    return events


def extract_markets(events: list) -> list:
    """Flatten nested markets from events; keep only those with price data."""
    markets = []
    for ev in events:
        for m in ev.get('markets', []):
            bid  = to_float(m.get('yes_bid_dollars'))
            ask  = to_float(m.get('yes_ask_dollars'))
            last = to_float(m.get('last_price_dollars'))
            vol  = float(m.get('volume_fp') or 0)
            if vol > 0 and (bid or ask or last):
                m['_event_title'] = ev.get('title', '')
                markets.append(m)
    return markets


def fetch_market_history(ticker: str, event_ticker: str, days: int) -> list:
    """Return list of daily close prices in (0, 1) over the last `days` days."""
    try:
        series_ticker = event_ticker.split('-')[0] if event_ticker else ''
        if not series_ticker:
            return []
        end_ts   = int(time.time())
        start_ts = end_ts - 86400 * days
        data = kalshi_get(
            f'series/{series_ticker}/markets/{ticker}/candlesticks',
            params={'start_ts': start_ts, 'end_ts': end_ts, 'period_interval': 1440},
        )
        prices = []
        for c in data.get('candlesticks', []):
            close = to_float((c.get('price') or {}).get('close_dollars'))
            if close is not None and 0 < close < 1:
                prices.append(close)
        return prices
    except Exception:
        return []


def midpoint(m: dict) -> Optional[float]:
    """Best-available price estimate: mid of bid/ask, falling back to last trade."""
    bid = to_float(m.get('yes_bid_dollars'))
    ask = to_float(m.get('yes_ask_dollars'))
    if bid is not None and ask is not None and ask > 0:
        return (bid + ask) / 2
    return to_float(m.get('last_price_dollars'))


def spread_pct(m: dict) -> float:
    """Bid-ask spread as a fraction of the mid-price."""
    bid = to_float(m.get('yes_bid_dollars'))
    ask = to_float(m.get('yes_ask_dollars'))
    if not bid or not ask or ask == 0:
        return 0.0
    mid = (bid + ask) / 2
    return (ask - bid) / mid if mid else 0.0


print('API helpers defined.')

## 2. Signal Estimation (Lectures 5.6 + 5.7)

**μ̂ (Lecture 5.6 — Return Predictability):**  
OLS slope of daily price changes over the lookback window, annualized by ×252.
Negated because prediction market prices mean-revert — a positive trend forecasts a downward return.

**RV (Lecture 5.7 — Realized Variance):**  
Variance of daily price changes over the lookback window, annualized by ×252.
Used to scale the position inversely — reduce exposure when the market has been noisy.

**R² filter:**  
A low R² means the regression slope is noise, not a forecast. Only signals with R² ≥ 0.10 are traded.

In [ ]:
def estimate(prices: list) -> Optional[dict]:
    """
    Estimate μ̂ and RV from a price history.

    Returns a dict with:
      mu_hat : negated OLS slope of daily changes, annualized (Lecture 5.6)
      rv     : realized variance of daily changes, annualized  (Lecture 5.7)
      r2     : OLS R² — quality filter on the signal
      n      : number of price observations
    Returns None if there is insufficient data or RV is effectively zero.
    """
    n = len(prices)
    if n < MIN_HISTORY:
        return None

    changes = np.diff(np.array(prices, dtype=float))  # daily price changes (Δp)
    if len(changes) < 2:
        return None

    # Lecture 5.6: OLS regression of changes on time index
    x = np.arange(len(changes), dtype=float)
    slope, _, r_val, _, _ = stats.linregress(x, changes)
    r2 = r_val ** 2

    # Lecture 5.7: realized variance, annualized
    rv = float(np.var(changes, ddof=1)) * 252
    if rv < 1e-8:
        return None   # degenerate — market hasn't moved at all

    return {
        'mu_hat': -slope * 252,  # negated: bet on mean-reversion, not momentum
        'rv':      rv,
        'r2':      r2,
        'n':       n,
    }


def mv_weight(mu_hat: float, rv: float, gamma: float, max_weight: float) -> float:
    """
    Mean-variance optimal weight: w = μ̂ / (γ · RV)

    Combines both lectures:
      - Numerator (μ̂):     invest proportionally to expected return (Lecture 5.6)
      - Denominator (γ·RV): scale down when variance is high         (Lecture 5.7)

    Capped at ±max_weight. At γ=2 the cap binds on virtually every trade,
    producing uniform position sizing — empirically better than vol-scaled
    sizing for this asset class (see design note 6).
    """
    return float(np.clip(mu_hat / (gamma * rv), -max_weight, max_weight))


print('Signal functions defined.')

## 3. Walk-Forward Backtest Engine

At each step `t` (after the lookback burn-in period):
1. Estimate μ̂ and RV from `prices[t-lookback : t]`
2. Apply R² and entry price filters
3. Compute weight `w = μ̂ / (γ · RV)`; skip if `w ≥ 0` (BUY YES trades excluded — see design note 11)
4. Record P&L over the next `hold_period` days

P&L per dollar of bankroll (three cost layers):
$$\text{net PnL} = \underbrace{w \cdot (p_{t+h} - p_t)}_{\text{raw}} - \underbrace{|w| \cdot \frac{\text{spread}}{2} \cdot p_t}_{\text{spread cost}} - \underbrace{0.07 \cdot |w| \cdot p_t}_{\text{Kalshi fee}}$$

**Kalshi fee derivation:** The fee formula is $0.07 \times C \times P \times (1-P)$, where $C$ is contracts
and $P$ is the YES price. For a BUY NO trade investing fraction $|w|$ of bankroll at YES price $P$:
the NO contract price is $(1-P)$, so $C = |w|/(1-P)$. Substituting:
$$0.07 \times \frac{|w|}{1-P} \times P \times (1-P) = 0.07 \times |w| \times P$$

In [ ]:
def simulate_market(
    ticker: str, title: str, prices: list, sprd: float,
    lookback: int, hold_period: int, gamma: float,
    max_weight: float, min_r2: float, min_price: float
) -> list:
    """
    Walk forward through a single market's price history, generating trades.
    Returns a list of trade dicts.
    """
    trades = []
    n = len(prices)

    for t in range(lookback, n - hold_period):
        hist = prices[t - lookback : t]
        est  = estimate(hist)

        # R² filter: skip if signal is unreliable
        if est is None or est['r2'] < min_r2:
            continue

        w = mv_weight(est['mu_hat'], est['rv'], gamma, max_weight)

        # Direction filter: only take BUY NO (contrarian bet against upward trends)
        if w >= 0:
            continue

        entry = prices[t]

        # Price filter: skip near-zero-probability markets (negative realized Sharpe)
        if entry < min_price:
            continue

        exit_ = prices[t + hold_period]

        # Raw P&L: w · ΔP (positive when price falls as forecasted for BUY NO)
        raw_pnl     = w * (exit_ - entry)

        # Spread cost: half-spread × |w| × entry price, deducted at entry
        spread_cost = abs(w) * sprd * entry / 2

        # Kalshi fee: 0.07 × C × P × (1-P), simplified to 0.07 × |w| × entry for BUY NO
        # Derivation: C = |w|/(1-entry), so fee = 0.07 × [|w|/(1-entry)] × entry × (1-entry)
        #             = 0.07 × |w| × entry
        kalshi_fee  = 0.07 * abs(w) * entry

        net_pnl = raw_pnl - spread_cost - kalshi_fee

        trades.append({
            'Ticker':     ticker,
            'Title':      title[:45],
            'Day':        t,
            'Entry P':    round(entry, 4),
            'Exit P':     round(exit_, 4),
            'Weight':     round(w, 4),
            'mu_hat':     round(est['mu_hat'], 4),
            'RV':         round(est['rv'], 4),
            'R2':         round(est['r2'], 3),
            'Raw PnL':    round(raw_pnl, 6),
            'Net PnL':    round(net_pnl, 6),
            'Spread':     round(sprd, 4),
            'Kalshi Fee': round(kalshi_fee, 6),
            'Win':        net_pnl > 0,
        })

    return trades


print('Backtest engine defined.')

## 4. Run the Backtest

Parameters used:
- γ=2, lookback=30d, hold=7d, fetch=180d, min-R²=0.10, min-price=0.15, limit=200

**Note:** This cell fetches live API data and will take 5–8 minutes to complete.

In [ ]:
# Fetch all open events and filter markets
events  = fetch_events()
markets = extract_markets(events)

# Volume filter
markets = [m for m in markets if float(m.get('volume_fp') or 0) >= 50]

# Sports exclusion: momentum dynamics make mean-reversion signal invalid
markets = [m for m in markets if not any(
    kw in (m.get('ticker') or '').lower() or kw in (m.get('title') or '').lower()
    for kw in SPORTS_KEYWORDS
)]

# Take top LIMIT markets by volume
markets.sort(key=lambda m: float(m.get('volume_fp') or 0), reverse=True)
markets = markets[:LIMIT]
print(f'Markets after volume + sports filter + limit: {len(markets)}')
print(f'Fetching {FETCH_DAYS}-day price history and simulating…\n')

# Walk-forward simulation
all_trades = []
total = len(markets)

for i, m in enumerate(markets):
    ticker       = m.get('ticker', '')
    event_ticker = m.get('event_ticker', '')
    title        = m.get('title') or ticker
    sprd         = spread_pct(m)

    if sprd > MAX_SPREAD:
        continue

    prices = fetch_market_history(ticker, event_ticker, days=FETCH_DAYS)
    time.sleep(0.15)   # respect API rate limit

    if len(prices) < LOOKBACK + HOLD_PERIOD + 1:
        continue

    trades = simulate_market(
        ticker=ticker, title=title, prices=prices, sprd=sprd,
        lookback=LOOKBACK, hold_period=HOLD_PERIOD,
        gamma=GAMMA, max_weight=MAX_WEIGHT,
        min_r2=MIN_R2, min_price=MIN_PRICE,
    )
    all_trades.extend(trades)

    if (i + 1) % 20 == 0:
        print(f'  {i+1}/{total} markets processed — {len(all_trades)} trades so far')

df = pd.DataFrame(all_trades)
print(f'\nSimulation complete: {len(df)} trades across {df["Ticker"].nunique()} markets.')

## 5. Results
### 5.1 Overall Backtest Statistics

In [ ]:
net = df['Net PnL']
raw = df['Raw PnL']

n_trades    = len(df)
n_markets   = df['Ticker'].nunique()
win_rate    = df['Win'].mean() * 100
mean_raw    = raw.mean()
mean_net    = net.mean()
std_net     = net.std(ddof=1)
ann_factor  = np.sqrt(252 / HOLD_PERIOD)
sharpe_raw  = (raw.mean() / raw.std(ddof=1)) * ann_factor if raw.std(ddof=1) > 1e-8 else 0.0
sharpe_net  = (mean_net  / std_net)           * ann_factor if std_net          > 1e-8 else 0.0
cumulative  = net.cumsum()
max_dd      = (cumulative - cumulative.cummax()).min()
gross_win   = net[net > 0].sum()
gross_loss  = net[net < 0].abs().sum()
pf          = gross_win / gross_loss if gross_loss > 1e-8 else float('inf')

summary = [
    ['Trades',                    f'{n_trades}'],
    ['Markets',                   f'{n_markets}'],
    ['Hold period',               f'{HOLD_PERIOD} day(s)'],
    ['Win rate',                  f'{win_rate:.1f}%  ({df["Win"].sum()}/{n_trades})'],
    ['Mean raw P&L / trade',      f'{mean_raw:+.6f}'],
    ['Mean net P&L / trade',      f'{mean_net:+.6f}  (after spread + Kalshi fee)'],
    ['Profit factor',             f'{pf:.2f}x'],
    ['Max drawdown (cumulative)', f'{max_dd:+.6f}'],
    ['Annualized Sharpe (raw)',   f'{sharpe_raw:+.3f}'],
    ['Annualized Sharpe (net)',   f'{sharpe_net:+.3f}'],
]
print(tabulate(summary, headers=['Metric', 'Value'], tablefmt='rounded_outline'))

### 5.2 Empirical Results

A point estimate of Sharpe is almost meaningless at 50–100 trades. We report:
- **t-test** of H₀: mean net PnL = 0
- **95% confidence interval** for mean net PnL
- **Bootstrap 95% CI for Sharpe** (Sharpe's sampling distribution is non-Normal at small n)
- **Cost breakdown**: spread drag vs. Kalshi fee drag

In [ ]:
# t-test: is mean net PnL distinguishable from zero?
t_stat, p_val = stats.ttest_1samp(net, 0)
ci_lo, ci_hi  = stats.t.interval(0.95, df=n_trades - 1, loc=mean_net, scale=stats.sem(net))

# Bootstrap Sharpe CI: resample trades with replacement 2,000 times
rng = np.random.default_rng(42)
bs_sharpes = []
for _ in range(2000):
    samp = rng.choice(net.values, size=n_trades, replace=True)
    sd   = samp.std(ddof=1)
    if sd > 1e-8:
        bs_sharpes.append((samp.mean() / sd) * ann_factor)
sh_lo, sh_hi = np.percentile(bs_sharpes, [2.5, 97.5])

# Cost breakdown
mean_fee    = df['Kalshi Fee'].mean()
mean_spread = (raw - net - df['Kalshi Fee']).mean()   # raw - fee - net = spread cost
total_drag  = mean_raw - mean_net
pct_drag    = (total_drag / mean_raw * 100) if abs(mean_raw) > 1e-8 else float('nan')

sig = '***' if p_val < 0.01 else ('**' if p_val < 0.05 else ('*' if p_val < 0.10 else 'ns'))

empirical = [
    ['t-statistic (H₀: μ_net = 0)',  f'{t_stat:+.3f}'],
    ['p-value (two-tailed)',          f'{p_val:.4f}  {sig}'],
    ['95% CI — mean net PnL (lower)', f'{ci_lo:+.6f}'],
    ['95% CI — mean net PnL (upper)', f'{ci_hi:+.6f}'],
    ['Bootstrap Sharpe CI (2.5%)',    f'{sh_lo:+.3f}'],
    ['Bootstrap Sharpe CI (97.5%)',   f'{sh_hi:+.3f}'],
    ['─── Cost breakdown ───',        ''],
    ['Spread drag per trade',         f'{mean_spread:+.6f}'],
    ['Kalshi fee drag per trade',     f'{mean_fee:+.6f}  (0.07 × |w| × entry)'],
    ['Total cost drag',               f'{total_drag:+.6f}  ({pct_drag:.1f}% of gross PnL)'],
]
print(tabulate(empirical, headers=['Metric', 'Value'], tablefmt='rounded_outline'))

### 5.3 Figures

In [ ]:
fig = plt.figure(figsize=(15, 9))
fig.suptitle(
    f'Foundation Strategy — Walk-Forward Backtest\n'
    f'γ={GAMMA}  lookback={LOOKBACK}d  hold={HOLD_PERIOD}d  '
    f'fetch={FETCH_DAYS}d  min-R²={MIN_R2}  min-price={MIN_PRICE}  limit={LIMIT}',
    fontsize=10, fontweight='bold'
)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.48, wspace=0.35)

# ── Figure 1: Cumulative net P&L ──────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
cum = net.cumsum().values
ax1.plot(cum, color='steelblue', linewidth=1.5)
ax1.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.6)
ax1.fill_between(range(len(cum)), cum, 0,
                 where=(cum >= 0), alpha=0.15, color='green', label='Profit')
ax1.fill_between(range(len(cum)), cum, 0,
                 where=(cum <  0), alpha=0.15, color='red',   label='Loss')
ax1.set_title('Fig 1: Cumulative Net P&L\n(after spread + Kalshi fee)', fontweight='bold')
ax1.set_xlabel('Trade #')
ax1.set_ylabel('Cumulative Net PnL')
ax1.legend(fontsize=7)

# ── Figure 2: Vol quintile mean net P&L (Lecture 5.7 check) ──────────────
ax2 = fig.add_subplot(gs[0, 1])
df_q = df.copy()
df_q['RV_Q'] = pd.qcut(df_q['RV'], q=5, labels=['Q1','Q2','Q3','Q4','Q5'])
q_means = df_q.groupby('RV_Q')['Net PnL'].mean()
q_wins  = df_q.groupby('RV_Q')['Win'].mean() * 100
colors  = ['#2ca02c' if v >= 0 else '#d62728' for v in q_means.values]
bars = ax2.bar(q_means.index.astype(str), q_means.values, color=colors, alpha=0.75,
               edgecolor='black', linewidth=0.5)
ax2.axhline(0, color='black', linewidth=0.8)
for bar, win in zip(bars, q_wins.values):
    yoff = max(q_means.values) * 0.03
    ax2.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + (yoff if bar.get_height() >= 0 else -yoff * 3),
             f'{win:.0f}%', ha='center', va='bottom', fontsize=7)
ax2.set_title('Fig 2: Mean Net P&L by Vol Quintile\n(Q1=lowest vol; Lecture 5.7 check)', fontweight='bold')
ax2.set_xlabel('Realized Variance Quintile')
ax2.set_ylabel('Mean Net PnL')

# ── Figure 3: Mean net P&L by entry price bucket ──────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
p40, p60 = df['Entry P'].quantile(0.40), df['Entry P'].quantile(0.60)
buckets = [
    (f'Low\n(P<{p40:.2f})',               df[df['Entry P'] <  p40]),
    (f'Mid\n({p40:.2f}–{p60:.2f})',        df[(df['Entry P'] >= p40) & (df['Entry P'] < p60)]),
    (f'High\n(P≥{p60:.2f})',               df[df['Entry P'] >= p60]),
]
labels_ep = [b[0] for b in buckets]
means_ep  = [b[1]['Net PnL'].mean() for b in buckets]
wins_ep   = [b[1]['Win'].mean() * 100 for b in buckets]
ns_ep     = [len(b[1]) for b in buckets]
colors_ep = ['#2ca02c' if v >= 0 else '#d62728' for v in means_ep]
bars3 = ax3.bar(labels_ep, means_ep, color=colors_ep, alpha=0.75,
                edgecolor='black', linewidth=0.5)
ax3.axhline(0, color='black', linewidth=0.8)
for bar, win, n_b in zip(bars3, wins_ep, ns_ep):
    ypos = bar.get_height() + abs(max(means_ep)) * 0.03
    ax3.text(bar.get_x() + bar.get_width()/2, ypos,
             f'{win:.0f}% win\n(n={n_b})', ha='center', va='bottom', fontsize=7)
ax3.set_title('Fig 3: Mean Net P&L by Entry Price\n(Key empirical finding)', fontweight='bold')
ax3.set_ylabel('Mean Net PnL')

# ── Figure 4: P&L distribution ────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 0])
ax4.hist(net.values, bins=25, color='steelblue', alpha=0.7,
         edgecolor='black', linewidth=0.4)
ax4.axvline(0,              color='black',  linewidth=0.9, linestyle='--', alpha=0.7, label='Zero')
ax4.axvline(mean_net,       color='red',    linewidth=1.5, linestyle='--',
            label=f'Mean={mean_net:.4f}')
ax4.axvline(np.median(net), color='orange', linewidth=1.2, linestyle=':',
            label=f'Median={np.median(net):.4f}')
ax4.set_title('Fig 4: Net P&L Distribution', fontweight='bold')
ax4.set_xlabel('Net PnL per Trade')
ax4.set_ylabel('Frequency')
ax4.legend(fontsize=7)

# ── Figure 5: Temporal stability — first vs. second half ──────────────────
ax5 = fig.add_subplot(gs[1, 1])
mid = len(df) // 2
halves = {'First half': df.iloc[:mid]['Net PnL'], 'Second half': df.iloc[mid:]['Net PnL']}
h_means = {k: v.mean() for k, v in halves.items()}
h_srs   = {k: (v.mean() / v.std(ddof=1)) * ann_factor
           for k, v in halves.items() if v.std(ddof=1) > 1e-8}
h_colors = ['#2ca02c' if v >= 0 else '#d62728' for v in h_means.values()]
bars5 = ax5.bar(h_means.keys(), h_means.values(), color=h_colors, alpha=0.75,
                edgecolor='black', linewidth=0.5)
ax5.axhline(0, color='black', linewidth=0.8)
for bar, (k, sr) in zip(bars5, h_srs.items()):
    _, pv = stats.ttest_1samp(halves[k], 0)
    sig_h = '***' if pv < 0.01 else ('**' if pv < 0.05 else ('*' if pv < 0.10 else 'ns'))
    ypos  = bar.get_height() + abs(max(h_means.values())) * 0.03
    ax5.text(bar.get_x() + bar.get_width()/2, ypos,
             f'SR={sr:.2f}\n({sig_h})', ha='center', va='bottom', fontsize=7)
ax5.set_title('Fig 5: Temporal Stability\n(First vs. second half of trades)', fontweight='bold')
ax5.set_ylabel('Mean Net PnL')

# ── Figure 6: Sharpe vs. min-R² threshold (robustness) ───────────────────
ax6 = fig.add_subplot(gs[1, 2])
r2_thresholds = [0.05, 0.10, 0.15, 0.20]
r2_result = []
for thresh in r2_thresholds:
    sub = df[df['R2'] >= thresh]
    if len(sub) >= 2:
        sd_s = sub['Net PnL'].std(ddof=1)
        sr_s = (sub['Net PnL'].mean() / sd_s) * ann_factor if sd_s > 1e-8 else 0.0
        r2_result.append((thresh, sr_s, len(sub)))
    else:
        r2_result.append((thresh, np.nan, 0))
valid = [(t, s, n) for t, s, n in r2_result if not np.isnan(s)]
if valid:
    ts, ss, ns_r = zip(*valid)
    ax6.plot([str(t) for t in ts], ss, marker='o', color='steelblue', linewidth=1.5)
    ax6.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.6)
    for t_v, s_v, n_v in valid:
        ax6.text(str(t_v), s_v + 0.04, f'n={n_v}', ha='center', fontsize=7)
ax6.set_title('Fig 6: Sharpe vs. min-R² Threshold\n(Robustness check)', fontweight='bold')
ax6.set_xlabel('min-R² cutoff')
ax6.set_ylabel('Annualized Sharpe (net)')

plt.savefig('backtest_figures.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → backtest_figures.png')

## 6. Detailed Tables
### Table 1: Volatility Quintile P&L

Tests the Lecture 5.7 mechanism: if vol-timing works, lower RV quintiles should have better
risk-adjusted returns. In prediction markets the relationship is inverted — high-volatility
markets show the best mean-reversion (design note 6).

In [ ]:
df_q = df.copy()
df_q['RV_Q'] = pd.qcut(df_q['RV'], q=5, labels=['Q1','Q2','Q3','Q4','Q5'])
q_table = df_q.groupby('RV_Q').agg(
    Avg_RV   = ('RV',      'mean'),
    Avg_Vol  = ('RV',      lambda x: np.sqrt(x.mean())),
    Avg_W    = ('Weight',  lambda x: x.abs().mean()),
    Mean_Net = ('Net PnL', 'mean'),
    Win_Rate = ('Win',     lambda x: x.mean() * 100),
    N        = ('Ticker',  'count'),
).reset_index()

rows = []
for _, r in q_table.iterrows():
    rows.append([
        r['RV_Q'], f"{r['Avg_RV']:.4f}", f"{r['Avg_Vol']:.1%}",
        f"{r['Avg_W']:.4f}", f"{r['Mean_Net']:+.6f}",
        f"{r['Win_Rate']:.1f}%", int(r['N'])
    ])
print(tabulate(rows,
               headers=['Quintile','Avg RV','Avg Vol','Avg |w|','Mean Net PnL','Win %','N'],
               tablefmt='rounded_outline'))

### Table 2: Net P&L Percentiles

In [ ]:
pcts = [5, 10, 25, 50, 75, 90, 95]
vals = np.percentile(net, pcts)
dist_rows = [[f'p{p}', f'{v:+.6f}'] for p, v in zip(pcts, vals)]
print(tabulate(dist_rows, headers=['Percentile', 'Net PnL'], tablefmt='rounded_outline'))

### Table 3: Sub-Sample Analysis

Tests whether results hold across different slices of the data.
A strategy whose edge lives in only one slice is fragile.

In [ ]:
def sub_stats(sub):
    """Return (n, win%, mean_net, sharpe, t_stat, p_val) for a trade sub-sample."""
    n = len(sub)
    if n < 2:
        return n, 0.0, float('nan'), float('nan'), 0.0, 1.0
    net_s = sub['Net PnL']
    mn    = net_s.mean()
    sd    = net_s.std(ddof=1)
    sr    = (mn / sd) * ann_factor if sd > 1e-8 else 0.0
    t, p  = stats.ttest_1samp(net_s, 0)
    win   = sub['Win'].mean() * 100
    return n, win, mn, sr, t, p


def sig_stars(p):
    return '***' if p < 0.01 else ('**' if p < 0.05 else ('*' if p < 0.10 else 'ns'))


mid = len(df) // 2
r2m = df['R2'].median()
p40, p60 = df['Entry P'].quantile(0.40), df['Entry P'].quantile(0.60)

sub_rows = []

# Temporal split
for label, sub in [('First half  (trades 1–N/2)', df.iloc[:mid]),
                   ('Second half (trades N/2–N)', df.iloc[mid:])]:
    n, win, mn, sr, t, p = sub_stats(sub)
    sub_rows.append([label, n, f'{win:.1f}%', f'{mn:+.5f}', f'{sr:+.3f}',
                     f'{t:+.2f}', f'{p:.4f} {sig_stars(p)}'])

sub_rows.append(['─── R² split ───', '', '', '', '', '', ''])

# R² split
for label, sub in [(f'Low R²  (< {r2m:.3f})', df[df['R2'] <  r2m]),
                   (f'High R² (≥ {r2m:.3f})', df[df['R2'] >= r2m])]:
    n, win, mn, sr, t, p = sub_stats(sub)
    sub_rows.append([label, n, f'{win:.1f}%', f'{mn:+.5f}', f'{sr:+.3f}',
                     f'{t:+.2f}', f'{p:.4f} {sig_stars(p)}'])

sub_rows.append(['─── Entry price split ───', '', '', '', '', '', ''])

# Entry price split
for label, sub in [
    (f'Low   entry (P < {p40:.2f})',            df[df['Entry P'] <  p40]),
    (f'Mid   entry ({p40:.2f}–{p60:.2f})',       df[(df['Entry P'] >= p40) & (df['Entry P'] < p60)]),
    (f'High  entry (P ≥ {p60:.2f})',             df[df['Entry P'] >= p60]),
]:
    n, win, mn, sr, t, p = sub_stats(sub)
    sub_rows.append([label, n, f'{win:.1f}%', f'{mn:+.5f}', f'{sr:+.3f}',
                     f'{t:+.2f}', f'{p:.4f} {sig_stars(p)}'])

print(tabulate(sub_rows,
               headers=['Sub-sample', 'N', 'Win%', 'Mean Net', 'Sharpe', 't-stat', 'p-val'],
               tablefmt='rounded_outline'))

### Table 4: Robustness Checks

Tests whether results are sensitive to the parameter choices made in strategy design.
A result that only works at exactly the chosen parameters is fragile.

In [ ]:
rob_rows = []

# 4a: min-R² sensitivity
rob_rows.append(['─── Sensitivity to min-R² threshold ───', '', '', '', '', ''])
for thresh in [0.05, 0.10, 0.15, 0.20, 0.25]:
    sub = df[df['R2'] >= thresh]
    if len(sub) < 2:
        rob_rows.append([f'min-R² ≥ {thresh}', len(sub), '—', '—', '—', '(too few)'])
        continue
    n, win, mn, sr, t, p = sub_stats(sub)
    rob_rows.append([f'min-R² ≥ {thresh}', n, f'{win:.1f}%',
                     f'{mn:+.6f}', f'{sr:+.3f}', f'{p:.4f} {sig_stars(p)}'])

# 4b: Three-level cost comparison: gross / after spread / after spread + fee
rob_rows.append(['─── Cost-layer sensitivity ───', '', '', '', '', ''])
after_spread = raw - (raw - net - df['Kalshi Fee'])   # raw minus spread cost only
for label, series in [
    ('Gross (no costs)',              raw),
    ('After spread (no Kalshi fee)',  after_spread),
    ('Net   (spread + Kalshi fee)',   net),
]:
    mn_s = series.mean()
    sd_s = series.std(ddof=1)
    sr_s = (mn_s / sd_s) * ann_factor if sd_s > 1e-8 else 0.0
    _, p_s = stats.ttest_1samp(series, 0)
    rob_rows.append([label, len(series), f'{series.gt(0).mean()*100:.1f}%',
                     f'{mn_s:+.6f}', f'{sr_s:+.3f}', f'{p_s:.4f} {sig_stars(p_s)}'])

# 4c: Leave-one-market-out
rob_rows.append(['─── Leave-one-market-out ───', '', '', '', '', ''])
top_mkt   = df.groupby('Ticker')['Net PnL'].count().idxmax()
top_mkt_n = (df['Ticker'] == top_mkt).sum()
sub_loo   = df[df['Ticker'] != top_mkt]
if len(sub_loo) >= 2:
    n, win, mn, sr, t, p = sub_stats(sub_loo)
    rob_rows.append([f'Drop {top_mkt} ({top_mkt_n} trades)', n, f'{win:.1f}%',
                     f'{mn:+.6f}', f'{sr:+.3f}', f'{p:.4f} {sig_stars(p)}'])

print(tabulate(rob_rows,
               headers=['Check', 'N', 'Win%', 'Mean Net', 'Sharpe', 'p-val'],
               tablefmt='rounded_outline'))

### Table 5: Top Markets by Trade Count

In [ ]:
top_mkt_table = (
    df.groupby(['Ticker', 'Title'])
    .agg(
        Trades    = ('Win',     'count'),
        Win_Rate  = ('Win',     lambda x: x.mean() * 100),
        Mean_Net  = ('Net PnL', 'mean'),
        Total_Net = ('Net PnL', 'sum'),
    )
    .sort_values('Trades', ascending=False)
    .head(20)
    .reset_index()
)

rows = []
for _, r in top_mkt_table.iterrows():
    rows.append([
        r['Ticker'], r['Title'][:36], int(r['Trades']),
        f"{r['Win_Rate']:.1f}%",
        f"{r['Mean_Net']:+.6f}",
        f"{r['Total_Net']:+.6f}",
    ])

print(tabulate(rows,
               headers=['Ticker', 'Title', 'Trades', 'Win %', 'Mean Net', 'Total Net'],
               tablefmt='rounded_outline'))

total_pnl = df['Net PnL'].sum()
top2_pnl  = top_mkt_table.head(2)['Total_Net'].sum()
print(f'\nTotal strategy net PnL : {total_pnl:+.4f}')
print(f'Top 2 markets combined : {top2_pnl:+.4f}  ({top2_pnl/total_pnl*100:.0f}% of total)')
print('\n⚠ Survivorship bias: only currently-open markets are tested.')
print(f'  Costs: half-spread × |w| × entry  +  0.07 × |w| × entry (Kalshi fee).')
print(f'  Annualized Sharpe assumes {int(252/HOLD_PERIOD)} non-overlapping periods/year.')